# RAG (Retrieval-Augmented Generation): A Concise Tutorial

## 1. What Is RAG?

RAG combines information retrieval with a Large Language Model (LLM).

Instead of asking an LLM to answer using only its trained knowledge, a
RAG application:

1.  Searches an external knowledge source.
2.  Retrieves relevant information.
3.  Adds that information to the prompt.
4.  Asks the LLM to generate an evidence-based answer.

RAG is useful when information is private, frequently updated, or too
specific to be included in model training.

**Important:** RAG usually does not train or fine-tune the LLM. It
provides external context at answer time.

------------------------------------------------------------------------

## 2. The Main RAG Architecture

RAG has two main workflows.

### 2.1 Indexing: Prepare the Data

``` text
Documents / APIs / Databases
          |
          v
      Ingestion
          |
          v
 Parsing and Cleaning
          |
          v
       Chunking
          |
          v
      Metadata
          |
          v
     Embeddings
          |
          v
   Vector Database
```

-   **Data ingestion:** Collect data from PDFs, Word files, websites,
    databases, or APIs.
-   **Parsing:** Extract readable text and structured content.
-   **Cleaning:** Remove noise, duplicates, and irrelevant content.
-   **Chunking:** Split large content into smaller meaningful pieces.
-   **Metadata:** Add information such as source, date, category, and
    document ID.
-   **Embedding:** Convert each chunk into a numerical vector.
-   **Indexing:** Store vectors and associated content for search.

### 2.2 Retrieval and Generation: Answer a Question

``` text
User Question
     |
     v
Query Processing
     |
     v
Query Embedding
     |
     v
Vector / Hybrid Search
     |
     v
Top-K Results
     |
     v
Reranking (optional)
     |
     v
Context Construction
     |
     v
LLM
     |
     v
Generated Answer
```

-   **Query processing:** Understand the question and identify entities
    or filters.
-   **Query embedding:** Convert the question into a vector.
-   **Retrieval:** Find relevant chunks.
-   **Filtering:** Narrow results using metadata.
-   **Reranking:** Reorder results by relevance.
-   **Context construction:** Select and organize the best information.
-   **Generation:** Ask the LLM to answer using the retrieved context.

------------------------------------------------------------------------

## 3. Important RAG Concepts

### LLM (Large Language Model)

A model that understands and generates text. Examples include GPT,
Claude, Gemini, and Llama.

### Embedding Model

Converts text into numerical vectors that represent semantic meaning. It
is used for search, not for generating the final answer.

Examples: BGE, E5, Sentence Transformers, and Jina Embeddings.

**Tip:** Query and document embeddings must be compatible. If the
embedding model or dimensions change, the indexed content usually needs
to be re-embedded.

### Vector

A list of numbers representing text in a semantic vector space.

``` text
"Company debt is 9 million TRY"
        |
        v
[0.12, -0.45, 0.78, 0.21, ...]
```

### Vector Database

Stores embeddings and supports similarity search.

Examples: Milvus, Qdrant, Pinecone, and Weaviate.

### Chunking

Splits documents into searchable pieces.

Common methods:

-   **Fixed-size chunking:** Split by a fixed token or character length.
-   **Overlapping chunking:** Keep some shared text between neighboring
    chunks.
-   **Semantic chunking:** Split at meaningful topic or paragraph
    boundaries.
-   **Hierarchical chunking:** Preserve parent-child structure, such as
    document → section → subsection.

**Tip:** Prefer meaningful chunks over arbitrary text fragments. Keep
headings, units, dates, and important context with each chunk.

### Metadata

Additional fields describing a chunk.

Example:

``` json
{
  "company_id": "ABC123",
  "category": "debt_analysis",
  "period": "2025",
  "source": "financial_service"
}
```

Metadata supports filtering, traceability, freshness checks, and access
control.

### Similarity Search

Finds vectors close to the query vector. Common measures include cosine
similarity, inner product, and Euclidean distance.

**Tip:** A similarity score is not a probability that the result is
correct.

### Top-K

The number of results returned by retrieval. For example, `top_k=5`
returns five candidate chunks.

**Tip:** A low K may miss useful information; a high K may add noise.
Tune it with test questions.

### Hybrid Search

Combines vector search with keyword search.

-   Vector search finds semantically similar content.
-   Keyword search finds exact words, codes, names, and identifiers.

Useful when exact terms such as account numbers, product codes, or
financial metrics matter.

### Reranking

Re-evaluates retrieved chunks and puts the most relevant ones first. It
improves ordering but cannot find information that the initial retrieval
missed.

### Query Rewriting

Rephrases a question to improve search.

Example:

-   Original: "How much does the company owe?"
-   Search query: "Total debt and liabilities of the company"

Preserve important constraints such as company ID and reporting period.

### Context Window

The maximum amount of input and output tokens a model can process in one
request. It includes instructions, user question, conversation history,
retrieved chunks, tool results, and generated answer.

**Tip:** More context is not always better. Send relevant evidence, not
every available document.

### Prompt Engineering

Writing instructions that define the model's role, task, rules, and
expected output.

Example:

``` text
Use only the supplied context for factual claims.
Do not invent missing values.
If the evidence is insufficient, say so.
Distinguish source facts from your interpretation.
```

### Grounding and Citations

Grounding means basing an answer on retrieved evidence. Citations help
users trace an answer to its source.

### Hallucination

When a model produces unsupported or fabricated information.

Reduce risk with reliable retrieval, source citations, clear
instructions, validation, and abstention rules. No prompt can guarantee
zero hallucinations.

### Context Pollution

When irrelevant, duplicated, outdated, or conflicting chunks are sent to
the LLM.

Reduce it with filtering, deduplication, reranking, and context limits.

### Stale Data

Information that is outdated or no longer valid.

Store both the business-effective date and the time the data was
ingested or updated.

### Prompt Injection

Malicious instructions in user input or retrieved content that try to
change model behavior.

Treat retrieved documents as untrusted data. Do not let document text
override system instructions or grant access to tools.

------------------------------------------------------------------------

## 4. RAG Architecture Types

### 4.1 Naive RAG

The simplest design:

``` text
Question → Embedding → Vector Search → Context → LLM → Answer
```

Easy to build, but may struggle with ambiguous questions, complex
retrieval, and noisy results.

### 4.2 Advanced RAG

Adds improvements such as query rewriting, metadata filtering, hybrid
search, reranking, and context compression.

``` text
Question
   |
Query Analysis
   |
Hybrid Retrieval
   |
Filtering and Reranking
   |
Context
   |
LLM
   |
Answer
```

### 4.3 Modular RAG

Separates RAG into independent components, such as ingestion, chunking,
embedding, retrieval, reranking, generation, and evaluation.

**Benefit:** Components can be tested, replaced, and improved
independently.

### 4.4 Graph RAG

Uses entities and relationships in a graph.

Example:

``` text
ABC Holding
    |
   owns
    |
    v
ABC Company
    |
  has debt
    |
    v
Financial Record
```

Useful for questions involving relationships, connected entities, and
multi-step reasoning. It is not necessary for every RAG project.

### 4.5 Agentic RAG

An agent or orchestrator decides which retrieval sources or tools to use
and may perform several steps.

Example:

``` text
User Question
     |
   Agent
     |
     +--> Search documents
     |
     +--> Call business API
     |
     +--> Run calculation
     |
     v
 Combine Results
     |
     v
    LLM
```

Use it when a question requires multiple tools or dynamic steps. For
predictable workflows, a normal deterministic pipeline may be simpler
and safer.

------------------------------------------------------------------------

## 5. General and Enterprise RAG Design

A basic RAG application may only need a document loader, chunker,
embedding model, vector database, retriever, and LLM.

A larger enterprise application may also need:

``` text
User
 |
 v
API Gateway
 |
 v
Authentication / Authorization
 |
 v
Orchestrator
 |
 v
Query Router
 |          \
 v           v
RAG        Business APIs
 |           |
 v           v
Vector DB  Structured Data
 |           |
 +-----+-----+
       |
       v
Context Builder
       |
       v
      LLM
       |
       v
Response Validation
       |
       v
      User
```

### Choose the right data source

Not every question should use vector search.

  -----------------------------------------------------------------------
  Question                            Suitable approach
  ----------------------------------- -----------------------------------
  "Explain the credit policy."        RAG over documents

  "What is the total debt?"           API or database query

  "Calculate the debt ratio."         Structured data + code

  "Assess the company's financial     APIs + calculations + RAG + LLM
  position."                          

  "Which companies are subsidiaries?" Relational or graph query
  -----------------------------------------------------------------------

**Rule of thumb:** - Use RAG for finding relevant document knowledge. -
Use APIs or SQL for exact structured values. - Use code for calculations
and business rules. - Use the LLM to explain and summarize evidence.

### Example: Financial RAG

Question:

> What is ABC Company's debt ratio in 2025?

The financial service returns:

``` json
{
  "total_debt": 9000000,
  "total_assets": 15000000,
  "currency": "TRY",
  "period": 2025
}
```

Calculate the ratio in code:

``` python
def debt_ratio(total_debt, total_assets):
    if total_assets <= 0:
        raise ValueError("Total assets must be positive")

    return total_debt / total_assets * 100

ratio = debt_ratio(9_000_000, 15_000_000)
print(ratio)  # 60.0
```

Formula:

``` text
Debt Ratio = Total Debt / Total Assets × 100
```

The LLM can explain the result, but the service should provide the
source values and code should perform the calculation.

The organization must define the exact financial formula. Different
business contexts may use different definitions.

### Enterprise tips

-   Enforce authentication and authorization in the application and data
    layers, not only in prompts.
-   Apply tenant, customer, and user permission filters before returning
    retrieved content.
-   Keep source IDs, timestamps, versions, reporting periods, and units.
-   Protect sensitive information in logs and traces.
-   Use timeouts, bounded retries, rate limits, and monitoring.
-   Keep an audit trail for sensitive data access and tool calls.
-   Test cross-customer data isolation.
-   Treat retrieved content as untrusted input.
-   Define data refresh, deletion, and re-indexing processes.

------------------------------------------------------------------------

## 6. Common Problems and Solutions

  -----------------------------------------------------------------------
  Problem         |        Meaning             |    Possible solution
  ----------------------- ----------------------- -----------------------
  Hallucination   |       Unsupported answer   |   Grounding, citations,
                                                  validation, abstention

  Retrieval failure  |     Relevant chunk not   |   Improve chunking,
                          found                   embeddings, filters,
                                                  hybrid search

  Context pollution  |     Irrelevant chunks in  |  Filtering,
                          prompt                  deduplication,
                                                  reranking

  Stale data    |          Outdated information |   Refresh and version
                                                  policies

  Prompt injection  |      Malicious instructions  |Treat retrieved content
                          in content              as untrusted

  Slow responses   |       High end-to-end latency |  Optimize retrieval,
                                                  caching, model serving

  High cost      |         Excessive tokens or  |   Context limits,
                          model calls             routing, caching
  -----------------------------------------------------------------------


------------------------------------------------------------------------

## 7. RAG Evaluation

Evaluate retrieval and generation separately.

### Retrieval metrics

  Metric        What it measures
  ------------- ------------------------------------------------------------
  Recall@K      Whether relevant results appear in the first K results
  Precision@K   How many of the first K results are relevant
  MRR           Position of the first relevant result
  NDCG          Quality of the ranking, considering relevance and position

Example:

If relevant evidence appears in the top five results for 80 of 100
questions:

``` text
Recall@5 = 80 / 100 = 80%
```

### Answer and context metrics

-   **Faithfulness:** Is the answer supported by the context?
-   **Answer relevance:** Does it answer the question?
-   **Context precision:** Is the retrieved context relevant?
-   **Context recall:** Does the context contain the required
    information?

Tools such as Ragas can help evaluate RAG applications. Automated scores
should be combined with human review for high-impact use cases.

### Operational metrics

Monitor: - End-to-end latency (p50 and p95) - Retrieval and reranking
latency - Token usage and cost - API and tool-call success rate - Empty
retrieval rate - Error and timeout rates - Data freshness - User
feedback

Create a fixed test dataset with questions, expected sources, expected
facts, and expected behavior. Run it whenever prompts, embeddings,
chunking, or retrieval settings change.

------------------------------------------------------------------------

## 8. Common Tools and Technologies

  Tool            Typical use
  --------------- ---------------------------------------------
  LangChain       LLM application components and integrations
  LlamaIndex      Data ingestion, indexing, and retrieval
  LangGraph       Stateful, multi-step agent workflows
  Milvus          Vector database
  Qdrant          Vector database
  Pinecone        Managed vector database
  Elasticsearch   Keyword, vector, and hybrid search
  Redis           Caching and fast data access
  FastAPI         Python API development
  Ragas           RAG evaluation
  Ollama          Local model execution
  vLLM            LLM serving and inference

These tools solve different problems. Select them based on requirements
rather than using every framework in one project.

------------------------------------------------------------------------

## 9. Practical Checklist

### Before building

-   [ ] Identify data sources and data owners.
-   [ ] Decide which questions need RAG and which need APIs or SQL.
-   [ ] Define security and access requirements.
-   [ ] Choose an embedding model and vector database.

### During indexing

-   [ ] Parse and clean source data.
-   [ ] Preserve document structure and record boundaries.
-   [ ] Choose suitable chunking and overlap.
-   [ ] Add useful metadata.
-   [ ] Track source, version, and update time.

### During retrieval

-   [ ] Extract entities and constraints.
-   [ ] Apply authorization filters.
-   [ ] Tune Top-K.
-   [ ] Test vector and hybrid search.
-   [ ] Add reranking if evaluation shows a benefit.
-   [ ] Remove duplicate and irrelevant context.

### Before production

-   [ ] Validate answers and source references.
-   [ ] Test hallucination and prompt-injection scenarios.
-   [ ] Measure retrieval and answer quality.
-   [ ] Monitor latency, cost, and failures.
-   [ ] Define refresh, deletion, and re-indexing processes.

------------------------------------------------------------------------

## 10. Key Takeaways

1.  RAG combines retrieval with LLM generation.
2.  Indexing prepares data; retrieval finds relevant information at
    question time.
3.  Chunking and metadata strongly affect retrieval quality.
4.  Embeddings enable semantic search; they do not generate answers.
5.  Hybrid search helps when both meaning and exact terms matter.
6.  Reranking improves the order of retrieved results.
7.  Top-K and context size should be measured and tuned.
8.  Use APIs or databases for exact structured values.
9.  Use deterministic code for calculations.
10. Use the LLM to explain evidence, not invent facts.
11. Security and authorization must be enforced outside the LLM.
12. Evaluate retrieval, generation, and production behavior separately.

**A good RAG system does more than generate fluent text: it finds the
right information, respects access rules, and produces answers that can
be checked.**


# Enterprise RAG Architecture: A Practical Engineering Tutorial

## 1. Introduction

Retrieval-Augmented Generation (RAG) is an architecture that combines
information retrieval with a Large Language Model (LLM). Instead of
relying only on knowledge encoded during model training, a RAG
application retrieves relevant information from external sources and
supplies it to the model as context when generating an answer.

RAG is especially useful in enterprise environments because business
knowledge is often private, frequently updated, distributed across
systems, and subject to access-control rules.

Typical use cases include:

-   Internal policy and procedure assistants
-   Financial analysis assistants
-   Customer-support knowledge assistants
-   Enterprise search
-   Regulatory and compliance question answering
-   Technical documentation assistants

**Core principle:** retrieval provides evidence, business services
provide authoritative structured data, deterministic tools perform exact
calculations, and the LLM interprets and communicates the result.

RAG does not normally retrain or fine-tune the LLM. It supplies relevant
external context at inference time.

------------------------------------------------------------------------

## 2. The Two Main RAG Pipelines

A RAG system is commonly divided into two pipelines:

1.  **Indexing (offline or ingestion pipeline):** prepares source data
    so it can be searched.
2.  **Retrieval and generation (online query pipeline):** finds relevant
    information and generates an answer.

Keeping these pipelines separate makes the system easier to scale, test,
refresh, and operate.

### 2.1 Indexing pipeline

``` text
Enterprise Data Sources
        |
        v
Data Ingestion
        |
        v
Parsing and Normalization
        |
        v
Cleaning / Enrichment
        |
        v
Chunking
        |
        v
Metadata Assignment
        |
        v
Embedding Generation
        |
        v
Vector / Search Database
```

#### 2.1.1 Data ingestion

Data ingestion brings information from source systems into the RAG
platform.

Possible enterprise sources include:

-   PDF, DOCX, PPTX, and HTML documents
-   Excel and CSV files
-   SQL and NoSQL databases
-   REST or gRPC services
-   CRM and ERP platforms
-   Document-management systems
-   Internal knowledge bases
-   Financial and risk-analysis services

Ingestion should record source identity, timestamps, versions, and
processing status. For sensitive systems, source permissions and data
classification should also be preserved.

#### 2.1.2 Parsing and normalization

Parsing extracts usable content from source formats. Normalization makes
equivalent values consistent.

For example, an API may return:

``` json
{
  "company": "ABC A.S.",
  "revenue": 15000000,
  "total_debt": 9000000,
  "cash": 2000000,
  "currency": "TRY",
  "period": "2025"
}
```

A normalized text representation could be:

``` text
Company: ABC A.S.
Reporting period: 2025
Revenue: 15,000,000 TRY
Total debt: 9,000,000 TRY
Cash and cash equivalents: 2,000,000 TRY
```

Do not blindly flatten every JSON object. Preserve record boundaries,
parent-child relationships, units, dates, currencies, and the meaning of
null or missing values. For highly structured records, a direct database
or service query may be more reliable than vector retrieval.

#### 2.1.3 Cleaning and enrichment

Typical preprocessing tasks include:

-   Removing duplicated headers and footers
-   Fixing encoding problems
-   Normalizing whitespace and date formats
-   Detecting language
-   Extracting tables and headings
-   Removing irrelevant boilerplate
-   Adding source and business metadata
-   Detecting duplicate or superseded documents

Avoid removing information that changes meaning, such as negative signs,
units, reporting periods, or table headings.

#### 2.1.4 Chunking

Chunking divides large content into smaller, retrievable units. A good
chunk should contain enough context to be understood independently while
remaining focused on one topic.

Common strategies:

**Fixed-size chunking:** splits text by a target token or character
count. It is simple but may split a sentence, table, or concept.

**Overlapping chunking:** neighboring chunks share some text to reduce
boundary-related context loss.

``` text
Chunk 1: tokens 1–500
Chunk 2: tokens 450–950
Chunk 3: tokens 900–1400
```

Excessive overlap creates duplicate results and increases storage and
context costs.

**Semantic chunking:** splits content at meaningful boundaries such as
paragraphs, sections, or topic changes.

**Structural or hierarchical chunking:** preserves document structure,
such as:

``` text
Company
  └── Financial Statements
       └── Liabilities
            ├── Short-term liabilities
            └── Long-term liabilities
```

For financial JSON, a useful strategy is often one chunk per meaningful
record or category, with the company, period, category, and units
repeated in each chunk. This avoids chunks that depend on missing parent
context.

Example:

``` text
Company: ABC A.S.
Company ID: ABC123
Data type: Financial data
Period: 2025
Category: Debt analysis

Total debt: 9,000,000 TRY
Short-term debt: 5,000,000 TRY
Long-term debt: 4,000,000 TRY

Source: Financial service
```

Chunk size should be selected empirically. Test several sizes and
overlap values against representative questions rather than adopting one
universal setting.

#### 2.1.5 Metadata

Metadata describes a chunk and enables filtering, authorization,
traceability, and lifecycle management.

Example:

``` json
{
  "chunk_id": "ABC123_2025_DEBT_01",
  "company_id": "ABC123",
  "company_name": "ABC A.S.",
  "document_type": "financial_data",
  "category": "debt_analysis",
  "reporting_period": "2025",
  "source_system": "financial_service",
  "source_record_id": "record-9812",
  "version": 2,
  "updated_at": "2026-01-15"
}
```

Useful metadata fields may include:

-   Tenant, organization, or business-unit ID
-   User or group access scope
-   Company or customer ID
-   Document type and category
-   Reporting period and effective date
-   Source system and source record ID
-   Version and update timestamp
-   Language and data classification

Metadata is not a replacement for authorization. Access policies must be
enforced by trusted application and data-access layers.

#### 2.1.6 Embeddings

An embedding model converts text into a numerical vector representing
semantic features.

Conceptual example:

``` text
"ABC A.S. has total debt of 9 million TRY."
              |
              v
[0.12, -0.45, 0.78, 0.21, ...]
```

Queries and documents can then be compared in vector space.

An embedding model is different from a generative LLM:

  -----------------------------------------------------------------------
  Embedding model                     Generative LLM
  ----------------------------------- -----------------------------------
  Produces vector representations     Produces text or other output

  Used for indexing and query         Used for reasoning and generation
  representation                      

  Supports semantic retrieval         Interprets context and creates
                                      answers
  -----------------------------------------------------------------------

Examples of embedding model families include BGE, E5, Sentence
Transformers, and Jina Embeddings.

The query and document embeddings must be compatible: use the same model
family and configuration, compatible dimensions, and the model's
required query/document prefixes where applicable. Changing the
embedding model generally requires re-embedding the indexed content.

#### 2.1.7 Vector databases and search indexes

A vector database stores embeddings and supports nearest-neighbor
search. Systems commonly used for vector retrieval include:

-   Milvus
-   Qdrant
-   Pinecone
-   Weaviate
-   Elasticsearch (which also supports lexical and hybrid search)

A record may contain an ID, vector, text, and scalar metadata. The
vector index is only one part of the retrieval system; metadata indexes
and payload storage are also important.

------------------------------------------------------------------------

### 2.2 Retrieval and generation pipeline

``` text
User Query
    |
    v
Authentication and Authorization
    |
    v
Query Understanding
    |
    v
Query Embedding / Query Transformation
    |
    v
Metadata Filtering + Vector / Hybrid Search
    |
    v
Candidate Chunks
    |
    v
Reranking
    |
    v
Context Construction
    |
    v
Prompt + Context
    |
    v
LLM Generation
    |
    v
Validation and Citation Checks
    |
    v
Response
```

#### 2.2.1 Query understanding

The application identifies what the user is asking for and extracts
relevant entities, constraints, and intent.

Example question:

> What was ABC A.S.'s debt position in 2025?

Possible structured representation:

``` json
{
  "company_id": "ABC123",
  "period": "2025",
  "topic": "debt_position",
  "intent": "financial_analysis"
}
```

Query understanding can use deterministic parsing, rules, an LLM, or a
combination. Do not rely on an LLM to invent or guess entity
identifiers.

#### 2.2.2 Query rewriting and expansion

A user may use wording that differs from the wording in the source data.

For example:

-   "How much does the company owe?"
-   "What are its liabilities?"
-   "What is its debt position?"

Query rewriting can create a clearer search query. Query expansion can
add useful synonyms or domain terms. These techniques should preserve
key constraints such as company ID, reporting period, and metric.

#### 2.2.3 Query embedding

The query is converted into a vector using the compatible embedding
model used for the indexed content.

#### 2.2.4 Similarity search

The query vector is compared with stored vectors. Common similarity or
distance measures include:

-   Cosine similarity
-   Inner product
-   Euclidean (L2) distance

The correct metric depends on the embedding model and index
configuration. A similarity score is not a probability that a result is
correct.

#### 2.2.5 Metadata filtering

Metadata filters narrow the search space.

Example:

``` python
results = vector_store.search(
    query_vector=query_embedding,
    filter={
        "company_id": "ABC123",
        "reporting_period": "2025"
    },
    limit=10
)
```

The syntax is illustrative; actual filter syntax depends on the
database.

In enterprise applications, filters for tenant, user permissions, and
data scope must be derived from trusted identity and authorization
services---not accepted directly from untrusted user input.

#### 2.2.6 Top-K retrieval

Top-K is the number of candidate results returned by retrieval.

``` python
results = vector_store.search(
    query_vector=query_embedding,
    limit=5
)
```

A small K can miss relevant evidence. A large K can introduce noise,
duplicates, latency, and excess token usage. Tune K using an evaluation
dataset.

#### 2.2.7 Hybrid search

Hybrid search combines semantic vector retrieval with lexical or keyword
retrieval.

Vector search is useful for semantic similarity. Keyword search is
useful for exact terms, codes, names, identifiers, and rare technical
expressions.

For example, a query for "2025 EBITDA" may benefit from both semantic
matching and exact matching of the term `EBITDA`.

A common approach is to merge rankings using Reciprocal Rank Fusion
(RRF), or another calibrated fusion method, and then rerank the
candidates.

#### 2.2.8 Reranking

A reranker evaluates query-document pairs and reorders the candidate
chunks by relevance.

A common pattern is:

1.  Retrieve a relatively broad candidate set efficiently.
2.  Rerank the candidates with a more precise model.
3.  Keep only the best evidence for context construction.

Reranking can improve ranking quality, but it cannot recover a document
that retrieval never returned.

#### 2.2.9 Context construction and compression

Context construction selects and organizes evidence for the LLM.

It should:

-   Remove duplicates
-   Preserve source and reporting-period information
-   Keep units and definitions
-   Include relevant headings or parent context
-   Prioritize high-quality evidence
-   Respect the model's context-window budget

Context compression can remove irrelevant wording while retaining facts
and qualifications. It must not remove important caveats, units, dates,
or source attribution.

#### 2.2.10 Generation

The LLM receives instructions, the user's question, and retrieved
context.

Example system instructions:

``` text
You are an enterprise financial analysis assistant.

Rules:
1. Use the supplied evidence for factual claims.
2. Do not invent missing financial values.
3. Distinguish source facts from your interpretation.
4. State the reporting period and currency.
5. If evidence is insufficient or conflicting, explain the limitation.
6. Do not disclose information outside the user's authorized scope.
7. Cite the source record or document when available.
```

Prompt instructions improve behavior but do not replace access controls,
validation, or deterministic business logic.

#### 2.2.11 Response validation

Before returning an answer, the application may validate:

-   Required fields and output schema
-   Source references
-   Numeric values against authoritative tool results
-   Reporting period and currency
-   Policy and access constraints
-   Whether the answer makes unsupported claims

For high-impact workflows, use explicit human review or approval where
required.

------------------------------------------------------------------------

## 3. Core RAG Concepts

### 3.1 Large Language Model (LLM)

An LLM is a model trained to process and generate language. It can
summarize, classify, explain, extract structured information, and use
tools when supported.

Examples include GPT, Claude, Gemini, and Llama model families.

Enterprise model selection should consider:

-   Data residency and privacy requirements
-   Deployment model (managed API, private cloud, or on-premises)
-   Quality in the required languages
-   Tool/function calling support
-   Context-window size
-   Latency and throughput
-   Cost and licensing
-   Operational support and model lifecycle

### 3.2 Embedding model

An embedding model maps text to vectors. It supports semantic retrieval,
not answer generation. Model choice should be evaluated on the actual
domain, language, terminology, and query distribution.

### 3.3 Chunking

Chunking determines the units the retrieval system can find. Good
chunking preserves meaning and context. For structured data,
record-aware chunking is often preferable to arbitrary character
splitting.

### 3.4 Metadata

Metadata enables filtering, provenance, access scoping, and freshness
management. Design metadata before indexing large volumes of data
because changing the schema later may require reprocessing.

### 3.5 Context window

The context window is the total amount of input and output tokens a
model can handle in one request. It includes system instructions,
conversation history, user query, retrieved evidence, tool results, and
generated output.

A larger context window does not mean that all available data should be
sent to the model. Relevant context is usually better than excessive
context.

### 3.6 Similarity search

Similarity search finds vectors close to the query vector according to a
selected distance or similarity metric. Similarity scores are ranking
signals, not correctness guarantees.

### 3.7 Top-K

Top-K controls how many candidates are retrieved. Tune it together with
filtering, reranking, context limits, and evaluation metrics.

### 3.8 Hybrid search

Hybrid search combines semantic and lexical retrieval. It is especially
useful for enterprise content containing exact identifiers, product
codes, legal clauses, financial metrics, and domain-specific terms.

### 3.9 Reranking

Reranking improves the order of retrieved candidates by scoring their
relevance to the actual query. It is commonly used after a faster
first-stage retrieval.

### 3.10 Query rewriting

Query rewriting transforms ambiguous or conversational questions into
search-friendly queries while preserving the user's intent and
constraints.

### 3.11 Grounding and citations

Grounding means basing factual claims on supplied evidence. Citations or
source references allow users and downstream systems to trace claims
back to documents, records, or service responses.

### 3.12 Hallucination

A hallucination is unsupported or fabricated model output. Reduce its
impact through evidence-based prompts, source citations, abstention
rules, deterministic calculations, output validation, and evaluation. No
prompt can guarantee zero hallucinations.

### 3.13 Context pollution

Context pollution occurs when irrelevant, duplicated, stale, or
contradictory information is passed to the model. It can reduce answer
quality and waste tokens. Deduplication, filtering, reranking, and
context budgeting help address it.

### 3.14 Stale data

Stale data is information that is no longer current or valid. Store both
the business-effective date and the ingestion/update timestamp. Define
refresh, expiration, and deletion policies for each source.

### 3.15 Prompt injection

Prompt injection is content that attempts to override the model's
instructions or manipulate tool use. Retrieved documents must be treated
as untrusted data, not as system instructions. Enforce permissions and
tool policies outside the LLM.

------------------------------------------------------------------------

## 4. RAG Architecture Patterns

### 4.1 Naive RAG

The simplest pattern:

``` text
User Query
    |
Query Embedding
    |
Vector Search
    |
Top-K Chunks
    |
LLM
    |
Answer
```

**Advantages** - Simple to implement - Few components - Useful for
prototypes and small knowledge bases

**Limitations** - Limited query understanding - Sensitive to chunk
quality - May return irrelevant results - Often lacks robust access,
freshness, and evaluation controls

### 4.2 Advanced RAG

Advanced RAG adds retrieval and context improvements:

``` text
User Query
    |
Query Analysis / Rewriting
    |
Metadata Filtering
    |
Hybrid Retrieval
    |
Reranking
    |
Context Construction
    |
LLM
    |
Validated Answer
```

Typical techniques include query expansion, hybrid search, metadata
filtering, reranking, context compression, and source attribution.

### 4.3 Modular RAG

Modular RAG separates the system into replaceable components:

``` text
RAG Application
  ├── Ingestion
  ├── Parser
  ├── Chunker
  ├── Embedding Service
  ├── Index Manager
  ├── Query Analyzer
  ├── Retriever
  ├── Reranker
  ├── Context Builder
  ├── Generator
  ├── Validator
  └── Evaluator
```

This separation supports independent testing, scaling, monitoring, and
replacement of components.

### 4.4 Graph RAG

Graph RAG represents entities and relationships as a graph. Nodes may
represent companies, people, products, contracts, or documents; edges
represent relationships such as owns, supplies, guarantees, or belongs
to.

Example:

``` text
ABC Holding
   |
   | owns
   v
ABC A.S.
   |
   | has
   v
Financial Statements
   |
   | contains
   v
Debt Records
```

Graph-based retrieval can help answer multi-hop relationship questions,
such as:

-   Which subsidiaries belong to a holding?
-   Which companies share an owner?
-   Which suppliers are connected to a particular business unit?

Graph RAG is not necessary for every project. Use it when relationships
and multi-hop queries are important enough to justify graph extraction,
storage, and maintenance.

### 4.5 Agentic RAG

Agentic RAG combines retrieval with an agent or orchestrator that can
choose tools and perform multiple steps.

Example flow:

``` text
User Request
    |
Agent / Orchestrator
    |
    +--> Search policy documents
    |
    +--> Call financial data service
    |
    +--> Call calculation tool
    |
    +--> Retrieve prior-period data
    |
    v
Combine Evidence
    |
    v
LLM Response
```

An agent may plan, call tools, inspect results, and decide what to do
next. Because agents can take actions, tool permissions, argument
validation, timeouts, retry limits, and audit logs are essential.

Agentic RAG is not automatically better than a deterministic workflow.
Prefer a predictable workflow when the required steps are known.

------------------------------------------------------------------------

## 5. Enterprise RAG Architecture

A production enterprise RAG system usually includes more than a vector
database and an LLM.

### 5.1 Reference architecture

``` text
                         USER / CLIENT
                               |
                               v
                         API GATEWAY
                               |
                               v
                    AUTHENTICATION / AUTHZ
                               |
                               v
                       AI ORCHESTRATOR
                               |
                         QUERY ANALYZER
                               |
                         INTENT / ROUTER
                    ___________|____________
                   |           |            |
                   v           v            v
               RAG SERVICE  BUSINESS APIs  TOOL SERVICES
                   |           |            |
                   v           v            v
              VECTOR DB   FINANCIAL DATA  CALCULATION
              / SEARCH      SERVICES       ENGINE
                   |           |            |
                   |___________|____________|
                               |
                               v
                        CONTEXT BUILDER
                               |
                               v
                              LLM
                               |
                               v
                    RESPONSE VALIDATION
                               |
                               v
                         RESPONSE API
                               |
                               v
                             USER
```

### 5.2 Component responsibilities

  -----------------------------------------------------------------------
  Component                           Responsibility
  ----------------------------------- -----------------------------------
  API Gateway                         Request routing, rate limits, and
                                      API-level controls

  Authentication                      Verifies user identity

  Authorization                       Determines which resources the user
                                      may access

  Orchestrator                        Coordinates the workflow

  Query Analyzer                      Extracts intent, entities, and
                                      constraints

  Router                              Selects retrieval, business APIs,
                                      or tools

  RAG Service                         Retrieves relevant unstructured or
                                      indexed content

  Business APIs                       Return authoritative structured
                                      business data

  Calculation Engine                  Performs deterministic calculations

  Context Builder                     Selects and formats evidence for
                                      the model

  LLM                                 Generates explanations and
                                      summaries

  Response Validator                  Checks schema, sources, values, and
                                      policy constraints

  Observability                       Captures traces, metrics, errors,
                                      and latency
  -----------------------------------------------------------------------

### 5.3 RAG versus direct API or database access

Not every question should be answered through vector search.

  -----------------------------------------------------------------------
  User question                       Recommended approach
  ----------------------------------- -----------------------------------
  What is the company's total debt?   Authoritative financial API or
                                      database query

  Explain the credit policy           RAG over policy documents

  Calculate the debt ratio            Structured data + deterministic
                                      calculation

  Assess the company's financial      Financial APIs + calculations +
  position                            relevant RAG context + LLM

  Which subsidiaries are related to   Relational or graph query
  this company?                       
  -----------------------------------------------------------------------

A useful design rule:

-   Use RAG for finding relevant explanatory or document-based
    knowledge.
-   Use APIs or SQL for exact, current, structured values.
-   Use deterministic code for arithmetic and business rules.
-   Use the LLM to interpret evidence and communicate results.

### 5.4 Example: financial analysis workflow

User question:

> Assess ABC A.S.'s financial position in 2025, calculate its debt
> ratio, and compare it with 2024.

Step 1 --- Extract the company and reporting periods.

``` json
{
  "company_id": "ABC123",
  "periods": [2024, 2025],
  "intent": "financial_comparison"
}
```

Step 2 --- Fetch authoritative structured data.

``` python
current = financial_service.get_data(
    company_id="ABC123",
    year=2025
)

previous = financial_service.get_data(
    company_id="ABC123",
    year=2024
)
```

Step 3 --- Validate required values, units, currency, and period.

Example current-period data:

``` json
{
  "total_debt": 9000000,
  "total_assets": 15000000,
  "currency": "TRY",
  "period": 2025
}
```

Step 4 --- Calculate with deterministic code.

``` python
def calculate_debt_ratio(total_debt: float, total_assets: float) -> float:
    if total_assets <= 0:
        raise ValueError("Total assets must be positive")

    return total_debt / total_assets * 100
```

For the example:

``` text
Debt ratio = 9,000,000 / 15,000,000 × 100
           = 60%
```

The definition must be agreed with the organization's finance
methodology. Different organizations may define leverage and debt ratios
differently. Do not silently substitute one formula for another.

Step 5 --- Retrieve relevant explanatory documents if needed, such as
internal financial-analysis methodology or risk definitions.

Step 6 --- Build a context containing source facts, calculated results,
definitions, and provenance.

Step 7 --- Ask the LLM to explain the result and compare periods.
Require it to distinguish measured facts from interpretation.

Step 8 --- Validate the answer against source values and return
citations or source references.

**Engineering principle:** the LLM should not be the source of truth for
exact financial values or arithmetic.

------------------------------------------------------------------------

## 6. Enterprise Challenges and Best Practices

### 6.1 Hallucination

**Problem:** the model generates information that is not supported by
the available evidence.

**Mitigations:** - Require evidence-based answers. - Provide source
references. - Define when the model must abstain. - Use APIs for
authoritative values. - Calculate numeric results in code. - Validate
critical outputs. - Evaluate with representative test questions.

### 6.2 Retrieval failure

**Problem:** relevant information exists but is not retrieved.

Possible causes: - Poor chunk boundaries - Incompatible or weak
embeddings - Incorrect metadata filters - Low Top-K - Domain terminology
mismatch - Missing query rewriting - Incorrect index configuration

Potential improvements: - Improve parsing and chunking - Test
domain-specific embedding models - Add lexical retrieval - Review
metadata filters - Tune Top-K - Add reranking - Create a retrieval
evaluation set

### 6.3 Context pollution

**Problem:** irrelevant or duplicated chunks enter the prompt.

Mitigations: - Filter by company, period, and category. - Deduplicate
overlapping results. - Rerank candidates. - Compress context
carefully. - Set a context budget. - Preserve critical qualifications
and units.

### 6.4 Stale data and lifecycle management

Store and distinguish: - Business-effective date - Reporting period -
Source update timestamp - Ingestion timestamp - Document version

Define refresh frequency, expiration, deletion, and re-indexing
procedures. A document uploaded today may describe a financial period
from several years ago.

### 6.5 Security and authorization

Enterprise RAG must enforce security at the application and data layers.

Best practices: - Authenticate every request. - Derive user and tenant
scope from trusted identity claims. - Apply authorization filters before
retrieval. - Enforce authorization again in business services. - Avoid
exposing sensitive content in logs and traces. - Encrypt data in transit
and at rest. - Apply retention and deletion policies. - Test
cross-tenant and cross-customer isolation. - Audit access to sensitive
records.

Do not rely on a prompt instruction such as "do not show other
customers' data" as an access-control mechanism.

### 6.6 Prompt injection

Retrieved documents and user input may contain malicious instructions.

Best practices: - Treat retrieved content as untrusted data. - Separate
system instructions from retrieved evidence. - Do not let retrieved text
grant permissions. - Validate tool names and arguments. - Enforce
authorization outside the LLM. - Limit tool capabilities and side
effects. - Log and review suspicious activity.

### 6.7 Reliability and resilience

Production systems should also consider: - Timeouts and bounded
retries - Circuit breakers for failing dependencies - Idempotency for
actions - Graceful degradation - Queue-based ingestion - Dead-letter
handling - Versioned prompts and indexes - Backups and recovery - Rate
limiting and concurrency controls

------------------------------------------------------------------------

## 7. RAG Evaluation and Observability

A production RAG system must be evaluated at both retrieval and
generation stages.

### 7.1 Retrieval metrics

  -----------------------------------------------------------------------
  Metric                              Meaning
  ----------------------------------- -----------------------------------
  Recall@K                            Fraction of relevant evidence
                                      retrieved within the first K
                                      results

  Precision@K                         Fraction of the first K results
                                      that are relevant

  MRR                                 Rewards placing the first relevant
                                      result near the top

  NDCG                                Measures ranking quality while
                                      considering relevance and position
  -----------------------------------------------------------------------

Example: if relevant evidence is found in the top five results for 80
out of 100 test questions:

``` text
Recall@5 = 80 / 100 = 0.80 = 80%
```

### 7.2 Generation and context metrics

  -----------------------------------------------------------------------
  Metric                              Meaning
  ----------------------------------- -----------------------------------
  Faithfulness                        Whether the answer is supported by
                                      the supplied context

  Answer relevance                    Whether the answer addresses the
                                      question

  Context precision                   How much retrieved context is
                                      relevant

  Context recall                      Whether the context contains the
                                      required evidence
  -----------------------------------------------------------------------

Frameworks such as Ragas can help evaluate RAG pipelines. Automated
scores should be supplemented with human review, especially for
financial, legal, regulatory, and other high-impact domains.

### 7.3 Operational metrics

Monitor: - End-to-end latency, including p50 and p95 - Retrieval and
reranking latency - LLM latency and token usage - Cost per request - API
and tool-call success rates - Empty or low-confidence retrieval rate -
Timeout and retry rates - Index freshness - User feedback -
Authorization failures

Use distributed tracing to connect a user request to retrieval calls,
business API calls, model calls, and validation results. Avoid recording
secrets or unnecessary sensitive data.

### 7.4 Build an evaluation dataset

Create a version-controlled dataset containing:

-   User question
-   Expected intent or route
-   Expected entities and filters
-   Relevant source documents or records
-   Expected answer facts
-   Expected abstention behavior, where applicable

Include normal, ambiguous, multi-hop, out-of-scope, stale-data, and
adversarial questions. Run the dataset after changes to chunking,
embeddings, prompts, retrieval settings, or models.

------------------------------------------------------------------------

## 8. Useful Technologies

The following are examples, not mandatory components.

  Technology      Typical role
  --------------- ---------------------------------------------------
  LangChain       Components and integrations for LLM applications
  LlamaIndex      Data ingestion, indexing, and retrieval workflows
  LangGraph       Stateful and multi-step agent workflows
  Milvus          Vector database and vector search
  Qdrant          Vector database and filtering
  Elasticsearch   Lexical, vector, and hybrid search
  Redis           Caching and fast data access
  FastAPI         Python API development
  Ragas           RAG evaluation
  Ollama          Local model execution
  vLLM            High-throughput LLM serving

Choose technologies based on requirements, operational skills, security
constraints, scale, and measured performance. Avoid adding frameworks
when a small, explicit implementation is easier to maintain.

------------------------------------------------------------------------

## 9. Practical Design Checklist

### Data and indexing

-   [ ] Identify authoritative source systems.
-   [ ] Define ingestion and refresh schedules.
-   [ ] Preserve source IDs, versions, dates, and units.
-   [ ] Normalize data without losing business meaning.
-   [ ] Select chunking strategies by content type.
-   [ ] Design metadata and access-scope fields.
-   [ ] Select and evaluate an embedding model.
-   [ ] Define re-indexing and deletion processes.

### Retrieval

-   [ ] Understand intent and extract entities.
-   [ ] Preserve company, tenant, and period constraints.
-   [ ] Apply authorization before returning evidence.
-   [ ] Tune Top-K using an evaluation dataset.
-   [ ] Evaluate vector and hybrid retrieval.
-   [ ] Add reranking if it improves measured quality.
-   [ ] Deduplicate and build a context budget.

### Generation

-   [ ] Use clear system instructions.
-   [ ] Separate facts from interpretation.
-   [ ] Require citations or source references where possible.
-   [ ] Define abstention behavior.
-   [ ] Validate structured output and numeric claims.
-   [ ] Treat retrieved content as untrusted input.

### Production

-   [ ] Add authentication and authorization.
-   [ ] Add timeouts, retries, and rate limits.
-   [ ] Monitor latency, cost, and failures.
-   [ ] Add tracing and audit logs.
-   [ ] Protect sensitive data in logs.
-   [ ] Test tenant isolation and prompt-injection defenses.
-   [ ] Maintain regression evaluation tests.

------------------------------------------------------------------------

## 10. Recommended Learning Path

A practical learning sequence for a software engineer:

1.  **RAG foundations:** LLMs, embeddings, chunking, metadata, vector
    databases, and similarity search.
2.  **Retrieval engineering:** Top-K, filtering, hybrid search, query
    rewriting, and reranking.
3.  **Enterprise data:** JSON normalization, record-aware chunking,
    source-of-truth design, data freshness, and authorization.
4.  **Orchestration and tools:** API integration, function calling,
    routing, deterministic calculations, and error handling.
5.  **Advanced architectures:** Modular RAG, Graph RAG, and Agentic RAG.
6.  **Production engineering:** evaluation, observability, latency,
    caching, security, and cost control.

Build a small end-to-end implementation first. Then improve one
component at a time and measure the impact with a fixed evaluation
dataset.

------------------------------------------------------------------------

## 11. Final Principles

1.  RAG is a complete data and application architecture, not just a
    vector database.
2.  Indexing quality strongly affects retrieval quality.
3.  Chunking should preserve meaning, structure, and provenance.
4.  Metadata is essential for filtering, traceability, and lifecycle
    management.
5.  Similarity scores are not confidence or correctness scores.
6.  Hybrid search is useful when exact terms and semantic meaning both
    matter.
7.  Reranking can improve candidate ordering but cannot recover missing
    candidates.
8.  Use authoritative APIs or databases for exact structured values.
9.  Use deterministic code for calculations and business rules.
10. Use the LLM to interpret evidence and communicate results.
11. Enforce authorization outside the LLM.
12. Evaluate retrieval, generation, security, and operations separately.

**The goal of enterprise RAG is not merely to generate fluent answers.
It is to produce useful, traceable, secure, and reliable answers from
the right business data.**


# Enterprise RAG Architecture: A Practical Engineering Tutorial

## 1. Introduction

Retrieval-Augmented Generation (RAG) is an architecture that combines
information retrieval with a Large Language Model (LLM). Instead of
relying only on knowledge encoded during model training, a RAG
application retrieves relevant information from external sources and
supplies it to the model as context when generating an answer.

RAG is especially useful in enterprise environments because business
knowledge is often private, frequently updated, distributed across
systems, and subject to access-control rules.

Typical use cases include:

-   Internal policy and procedure assistants
-   Financial analysis assistants
-   Customer-support knowledge assistants
-   Enterprise search
-   Regulatory and compliance question answering
-   Technical documentation assistants

**Core principle:** retrieval provides evidence, business services
provide authoritative structured data, deterministic tools perform exact
calculations, and the LLM interprets and communicates the result.

RAG does not normally retrain or fine-tune the LLM. It supplies relevant
external context at inference time.

------------------------------------------------------------------------

## 2. The Two Main RAG Pipelines

A RAG system is commonly divided into two pipelines:

1.  **Indexing (offline or ingestion pipeline):** prepares source data
    so it can be searched.
2.  **Retrieval and generation (online query pipeline):** finds relevant
    information and generates an answer.

Keeping these pipelines separate makes the system easier to scale, test,
refresh, and operate.

### 2.1 Indexing pipeline

``` text
Enterprise Data Sources
        |
        v
Data Ingestion
        |
        v
Parsing and Normalization
        |
        v
Cleaning / Enrichment
        |
        v
Chunking
        |
        v
Metadata Assignment
        |
        v
Embedding Generation
        |
        v
Vector / Search Database
```

#### 2.1.1 Data ingestion

Data ingestion brings information from source systems into the RAG
platform.

Possible enterprise sources include:

-   PDF, DOCX, PPTX, and HTML documents
-   Excel and CSV files
-   SQL and NoSQL databases
-   REST or gRPC services
-   CRM and ERP platforms
-   Document-management systems
-   Internal knowledge bases
-   Financial and risk-analysis services

Ingestion should record source identity, timestamps, versions, and
processing status. For sensitive systems, source permissions and data
classification should also be preserved.

#### 2.1.2 Parsing and normalization

Parsing extracts usable content from source formats. Normalization makes
equivalent values consistent.

For example, an API may return:

``` json
{
  "company": "ABC A.S.",
  "revenue": 15000000,
  "total_debt": 9000000,
  "cash": 2000000,
  "currency": "TRY",
  "period": "2025"
}
```

A normalized text representation could be:

``` text
Company: ABC A.S.
Reporting period: 2025
Revenue: 15,000,000 TRY
Total debt: 9,000,000 TRY
Cash and cash equivalents: 2,000,000 TRY
```

Do not blindly flatten every JSON object. Preserve record boundaries,
parent-child relationships, units, dates, currencies, and the meaning of
null or missing values. For highly structured records, a direct database
or service query may be more reliable than vector retrieval.

#### 2.1.3 Cleaning and enrichment

Typical preprocessing tasks include:

-   Removing duplicated headers and footers
-   Fixing encoding problems
-   Normalizing whitespace and date formats
-   Detecting language
-   Extracting tables and headings
-   Removing irrelevant boilerplate
-   Adding source and business metadata
-   Detecting duplicate or superseded documents

Avoid removing information that changes meaning, such as negative signs,
units, reporting periods, or table headings.

#### 2.1.4 Chunking

Chunking divides large content into smaller, retrievable units. A good
chunk should contain enough context to be understood independently while
remaining focused on one topic.

Common strategies:

**Fixed-size chunking:** splits text by a target token or character
count. It is simple but may split a sentence, table, or concept.

**Overlapping chunking:** neighboring chunks share some text to reduce
boundary-related context loss.

``` text
Chunk 1: tokens 1–500
Chunk 2: tokens 450–950
Chunk 3: tokens 900–1400
```

Excessive overlap creates duplicate results and increases storage and
context costs.

**Semantic chunking:** splits content at meaningful boundaries such as
paragraphs, sections, or topic changes.

**Structural or hierarchical chunking:** preserves document structure,
such as:

``` text
Company
  └── Financial Statements
       └── Liabilities
            ├── Short-term liabilities
            └── Long-term liabilities
```

For financial JSON, a useful strategy is often one chunk per meaningful
record or category, with the company, period, category, and units
repeated in each chunk. This avoids chunks that depend on missing parent
context.

Example:

``` text
Company: ABC A.S.
Company ID: ABC123
Data type: Financial data
Period: 2025
Category: Debt analysis

Total debt: 9,000,000 TRY
Short-term debt: 5,000,000 TRY
Long-term debt: 4,000,000 TRY

Source: Financial service
```

Chunk size should be selected empirically. Test several sizes and
overlap values against representative questions rather than adopting one
universal setting.

#### 2.1.5 Metadata

Metadata describes a chunk and enables filtering, authorization,
traceability, and lifecycle management.

Example:

``` json
{
  "chunk_id": "ABC123_2025_DEBT_01",
  "company_id": "ABC123",
  "company_name": "ABC A.S.",
  "document_type": "financial_data",
  "category": "debt_analysis",
  "reporting_period": "2025",
  "source_system": "financial_service",
  "source_record_id": "record-9812",
  "version": 2,
  "updated_at": "2026-01-15"
}
```

Useful metadata fields may include:

-   Tenant, organization, or business-unit ID
-   User or group access scope
-   Company or customer ID
-   Document type and category
-   Reporting period and effective date
-   Source system and source record ID
-   Version and update timestamp
-   Language and data classification

Metadata is not a replacement for authorization. Access policies must be
enforced by trusted application and data-access layers.

#### 2.1.6 Embeddings

An embedding model converts text into a numerical vector representing
semantic features.

Conceptual example:

``` text
"ABC A.S. has total debt of 9 million TRY."
              |
              v
[0.12, -0.45, 0.78, 0.21, ...]
```

Queries and documents can then be compared in vector space.

An embedding model is different from a generative LLM:

  -----------------------------------------------------------------------
  Embedding model                     Generative LLM
  ----------------------------------- -----------------------------------
  Produces vector representations     Produces text or other output

  Used for indexing and query         Used for reasoning and generation
  representation                      

  Supports semantic retrieval         Interprets context and creates
                                      answers
  -----------------------------------------------------------------------

Examples of embedding model families include BGE, E5, Sentence
Transformers, and Jina Embeddings.

The query and document embeddings must be compatible: use the same model
family and configuration, compatible dimensions, and the model's
required query/document prefixes where applicable. Changing the
embedding model generally requires re-embedding the indexed content.

#### 2.1.7 Vector databases and search indexes

A vector database stores embeddings and supports nearest-neighbor
search. Systems commonly used for vector retrieval include:

-   Milvus
-   Qdrant
-   Pinecone
-   Weaviate
-   Elasticsearch (which also supports lexical and hybrid search)

A record may contain an ID, vector, text, and scalar metadata. The
vector index is only one part of the retrieval system; metadata indexes
and payload storage are also important.

------------------------------------------------------------------------

### 2.2 Retrieval and generation pipeline

``` text
User Query
    |
    v
Authentication and Authorization
    |
    v
Query Understanding
    |
    v
Query Embedding / Query Transformation
    |
    v
Metadata Filtering + Vector / Hybrid Search
    |
    v
Candidate Chunks
    |
    v
Reranking
    |
    v
Context Construction
    |
    v
Prompt + Context
    |
    v
LLM Generation
    |
    v
Validation and Citation Checks
    |
    v
Response
```

#### 2.2.1 Query understanding

The application identifies what the user is asking for and extracts
relevant entities, constraints, and intent.

Example question:

> What was ABC A.S.'s debt position in 2025?

Possible structured representation:

``` json
{
  "company_id": "ABC123",
  "period": "2025",
  "topic": "debt_position",
  "intent": "financial_analysis"
}
```

Query understanding can use deterministic parsing, rules, an LLM, or a
combination. Do not rely on an LLM to invent or guess entity
identifiers.

#### 2.2.2 Query rewriting and expansion

A user may use wording that differs from the wording in the source data.

For example:

-   "How much does the company owe?"
-   "What are its liabilities?"
-   "What is its debt position?"

Query rewriting can create a clearer search query. Query expansion can
add useful synonyms or domain terms. These techniques should preserve
key constraints such as company ID, reporting period, and metric.

#### 2.2.3 Query embedding

The query is converted into a vector using the compatible embedding
model used for the indexed content.

#### 2.2.4 Similarity search

The query vector is compared with stored vectors. Common similarity or
distance measures include:

-   Cosine similarity
-   Inner product
-   Euclidean (L2) distance

The correct metric depends on the embedding model and index
configuration. A similarity score is not a probability that a result is
correct.

#### 2.2.5 Metadata filtering

Metadata filters narrow the search space.

Example:

``` python
results = vector_store.search(
    query_vector=query_embedding,
    filter={
        "company_id": "ABC123",
        "reporting_period": "2025"
    },
    limit=10
)
```

The syntax is illustrative; actual filter syntax depends on the
database.

In enterprise applications, filters for tenant, user permissions, and
data scope must be derived from trusted identity and authorization
services---not accepted directly from untrusted user input.

#### 2.2.6 Top-K retrieval

Top-K is the number of candidate results returned by retrieval.

``` python
results = vector_store.search(
    query_vector=query_embedding,
    limit=5
)
```

A small K can miss relevant evidence. A large K can introduce noise,
duplicates, latency, and excess token usage. Tune K using an evaluation
dataset.

#### 2.2.7 Hybrid search

Hybrid search combines semantic vector retrieval with lexical or keyword
retrieval.

Vector search is useful for semantic similarity. Keyword search is
useful for exact terms, codes, names, identifiers, and rare technical
expressions.

For example, a query for "2025 EBITDA" may benefit from both semantic
matching and exact matching of the term `EBITDA`.

A common approach is to merge rankings using Reciprocal Rank Fusion
(RRF), or another calibrated fusion method, and then rerank the
candidates.

#### 2.2.8 Reranking

A reranker evaluates query-document pairs and reorders the candidate
chunks by relevance.

A common pattern is:

1.  Retrieve a relatively broad candidate set efficiently.
2.  Rerank the candidates with a more precise model.
3.  Keep only the best evidence for context construction.

Reranking can improve ranking quality, but it cannot recover a document
that retrieval never returned.

#### 2.2.9 Context construction and compression

Context construction selects and organizes evidence for the LLM.

It should:

-   Remove duplicates
-   Preserve source and reporting-period information
-   Keep units and definitions
-   Include relevant headings or parent context
-   Prioritize high-quality evidence
-   Respect the model's context-window budget

Context compression can remove irrelevant wording while retaining facts
and qualifications. It must not remove important caveats, units, dates,
or source attribution.

#### 2.2.10 Generation

The LLM receives instructions, the user's question, and retrieved
context.

Example system instructions:

``` text
You are an enterprise financial analysis assistant.

Rules:
1. Use the supplied evidence for factual claims.
2. Do not invent missing financial values.
3. Distinguish source facts from your interpretation.
4. State the reporting period and currency.
5. If evidence is insufficient or conflicting, explain the limitation.
6. Do not disclose information outside the user's authorized scope.
7. Cite the source record or document when available.
```

Prompt instructions improve behavior but do not replace access controls,
validation, or deterministic business logic.

#### 2.2.11 Response validation

Before returning an answer, the application may validate:

-   Required fields and output schema
-   Source references
-   Numeric values against authoritative tool results
-   Reporting period and currency
-   Policy and access constraints
-   Whether the answer makes unsupported claims

For high-impact workflows, use explicit human review or approval where
required.

------------------------------------------------------------------------

## 3. Core RAG Concepts

### 3.1 Large Language Model (LLM)

An LLM is a model trained to process and generate language. It can
summarize, classify, explain, extract structured information, and use
tools when supported.

Examples include GPT, Claude, Gemini, and Llama model families.

Enterprise model selection should consider:

-   Data residency and privacy requirements
-   Deployment model (managed API, private cloud, or on-premises)
-   Quality in the required languages
-   Tool/function calling support
-   Context-window size
-   Latency and throughput
-   Cost and licensing
-   Operational support and model lifecycle

### 3.2 Embedding model

An embedding model maps text to vectors. It supports semantic retrieval,
not answer generation. Model choice should be evaluated on the actual
domain, language, terminology, and query distribution.

### 3.3 Chunking

Chunking determines the units the retrieval system can find. Good
chunking preserves meaning and context. For structured data,
record-aware chunking is often preferable to arbitrary character
splitting.

### 3.4 Metadata

Metadata enables filtering, provenance, access scoping, and freshness
management. Design metadata before indexing large volumes of data
because changing the schema later may require reprocessing.

### 3.5 Context window

The context window is the total amount of input and output tokens a
model can handle in one request. It includes system instructions,
conversation history, user query, retrieved evidence, tool results, and
generated output.

A larger context window does not mean that all available data should be
sent to the model. Relevant context is usually better than excessive
context.

### 3.6 Similarity search

Similarity search finds vectors close to the query vector according to a
selected distance or similarity metric. Similarity scores are ranking
signals, not correctness guarantees.

### 3.7 Top-K

Top-K controls how many candidates are retrieved. Tune it together with
filtering, reranking, context limits, and evaluation metrics.

### 3.8 Hybrid search

Hybrid search combines semantic and lexical retrieval. It is especially
useful for enterprise content containing exact identifiers, product
codes, legal clauses, financial metrics, and domain-specific terms.

### 3.9 Reranking

Reranking improves the order of retrieved candidates by scoring their
relevance to the actual query. It is commonly used after a faster
first-stage retrieval.

### 3.10 Query rewriting

Query rewriting transforms ambiguous or conversational questions into
search-friendly queries while preserving the user's intent and
constraints.

### 3.11 Grounding and citations

Grounding means basing factual claims on supplied evidence. Citations or
source references allow users and downstream systems to trace claims
back to documents, records, or service responses.

### 3.12 Hallucination

A hallucination is unsupported or fabricated model output. Reduce its
impact through evidence-based prompts, source citations, abstention
rules, deterministic calculations, output validation, and evaluation. No
prompt can guarantee zero hallucinations.

### 3.13 Context pollution

Context pollution occurs when irrelevant, duplicated, stale, or
contradictory information is passed to the model. It can reduce answer
quality and waste tokens. Deduplication, filtering, reranking, and
context budgeting help address it.

### 3.14 Stale data

Stale data is information that is no longer current or valid. Store both
the business-effective date and the ingestion/update timestamp. Define
refresh, expiration, and deletion policies for each source.

### 3.15 Prompt injection

Prompt injection is content that attempts to override the model's
instructions or manipulate tool use. Retrieved documents must be treated
as untrusted data, not as system instructions. Enforce permissions and
tool policies outside the LLM.

------------------------------------------------------------------------

## 4. RAG Architecture Patterns

### 4.1 Naive RAG

The simplest pattern:

``` text
User Query
    |
Query Embedding
    |
Vector Search
    |
Top-K Chunks
    |
LLM
    |
Answer
```

**Advantages** - Simple to implement - Few components - Useful for
prototypes and small knowledge bases

**Limitations** - Limited query understanding - Sensitive to chunk
quality - May return irrelevant results - Often lacks robust access,
freshness, and evaluation controls

### 4.2 Advanced RAG

Advanced RAG adds retrieval and context improvements:

``` text
User Query
    |
Query Analysis / Rewriting
    |
Metadata Filtering
    |
Hybrid Retrieval
    |
Reranking
    |
Context Construction
    |
LLM
    |
Validated Answer
```

Typical techniques include query expansion, hybrid search, metadata
filtering, reranking, context compression, and source attribution.

### 4.3 Modular RAG

Modular RAG separates the system into replaceable components:

``` text
RAG Application
  ├── Ingestion
  ├── Parser
  ├── Chunker
  ├── Embedding Service
  ├── Index Manager
  ├── Query Analyzer
  ├── Retriever
  ├── Reranker
  ├── Context Builder
  ├── Generator
  ├── Validator
  └── Evaluator
```

This separation supports independent testing, scaling, monitoring, and
replacement of components.

### 4.4 Graph RAG

Graph RAG represents entities and relationships as a graph. Nodes may
represent companies, people, products, contracts, or documents; edges
represent relationships such as owns, supplies, guarantees, or belongs
to.

Example:

``` text
ABC Holding
   |
   | owns
   v
ABC A.S.
   |
   | has
   v
Financial Statements
   |
   | contains
   v
Debt Records
```

Graph-based retrieval can help answer multi-hop relationship questions,
such as:

-   Which subsidiaries belong to a holding?
-   Which companies share an owner?
-   Which suppliers are connected to a particular business unit?

Graph RAG is not necessary for every project. Use it when relationships
and multi-hop queries are important enough to justify graph extraction,
storage, and maintenance.

### 4.5 Agentic RAG

Agentic RAG combines retrieval with an agent or orchestrator that can
choose tools and perform multiple steps.

Example flow:

``` text
User Request
    |
Agent / Orchestrator
    |
    +--> Search policy documents
    |
    +--> Call financial data service
    |
    +--> Call calculation tool
    |
    +--> Retrieve prior-period data
    |
    v
Combine Evidence
    |
    v
LLM Response
```

An agent may plan, call tools, inspect results, and decide what to do
next. Because agents can take actions, tool permissions, argument
validation, timeouts, retry limits, and audit logs are essential.

Agentic RAG is not automatically better than a deterministic workflow.
Prefer a predictable workflow when the required steps are known.

------------------------------------------------------------------------

## 5. Enterprise RAG Architecture

A production enterprise RAG system usually includes more than a vector
database and an LLM.

### 5.1 Reference architecture

``` text
                         USER / CLIENT
                               |
                               v
                         API GATEWAY
                               |
                               v
                    AUTHENTICATION / AUTHZ
                               |
                               v
                       AI ORCHESTRATOR
                               |
                         QUERY ANALYZER
                               |
                         INTENT / ROUTER
                    ___________|____________
                   |           |            |
                   v           v            v
               RAG SERVICE  BUSINESS APIs  TOOL SERVICES
                   |           |            |
                   v           v            v
              VECTOR DB   FINANCIAL DATA  CALCULATION
              / SEARCH      SERVICES       ENGINE
                   |           |            |
                   |___________|____________|
                               |
                               v
                        CONTEXT BUILDER
                               |
                               v
                              LLM
                               |
                               v
                    RESPONSE VALIDATION
                               |
                               v
                         RESPONSE API
                               |
                               v
                             USER
```

### 5.2 Component responsibilities

  -----------------------------------------------------------------------
  Component                           Responsibility
  ----------------------------------- -----------------------------------
  API Gateway                         Request routing, rate limits, and
                                      API-level controls

  Authentication                      Verifies user identity

  Authorization                       Determines which resources the user
                                      may access

  Orchestrator                        Coordinates the workflow

  Query Analyzer                      Extracts intent, entities, and
                                      constraints

  Router                              Selects retrieval, business APIs,
                                      or tools

  RAG Service                         Retrieves relevant unstructured or
                                      indexed content

  Business APIs                       Return authoritative structured
                                      business data

  Calculation Engine                  Performs deterministic calculations

  Context Builder                     Selects and formats evidence for
                                      the model

  LLM                                 Generates explanations and
                                      summaries

  Response Validator                  Checks schema, sources, values, and
                                      policy constraints

  Observability                       Captures traces, metrics, errors,
                                      and latency
  -----------------------------------------------------------------------

### 5.3 RAG versus direct API or database access

Not every question should be answered through vector search.

  -----------------------------------------------------------------------
  User question                       Recommended approach
  ----------------------------------- -----------------------------------
  What is the company's total debt?   Authoritative financial API or
                                      database query

  Explain the credit policy           RAG over policy documents

  Calculate the debt ratio            Structured data + deterministic
                                      calculation

  Assess the company's financial      Financial APIs + calculations +
  position                            relevant RAG context + LLM

  Which subsidiaries are related to   Relational or graph query
  this company?                       
  -----------------------------------------------------------------------

A useful design rule:

-   Use RAG for finding relevant explanatory or document-based
    knowledge.
-   Use APIs or SQL for exact, current, structured values.
-   Use deterministic code for arithmetic and business rules.
-   Use the LLM to interpret evidence and communicate results.

### 5.4 Example: financial analysis workflow

User question:

> Assess ABC A.S.'s financial position in 2025, calculate its debt
> ratio, and compare it with 2024.

Step 1 --- Extract the company and reporting periods.

``` json
{
  "company_id": "ABC123",
  "periods": [2024, 2025],
  "intent": "financial_comparison"
}
```

Step 2 --- Fetch authoritative structured data.

``` python
current = financial_service.get_data(
    company_id="ABC123",
    year=2025
)

previous = financial_service.get_data(
    company_id="ABC123",
    year=2024
)
```

Step 3 --- Validate required values, units, currency, and period.

Example current-period data:

``` json
{
  "total_debt": 9000000,
  "total_assets": 15000000,
  "currency": "TRY",
  "period": 2025
}
```

Step 4 --- Calculate with deterministic code.

``` python
def calculate_debt_ratio(total_debt: float, total_assets: float) -> float:
    if total_assets <= 0:
        raise ValueError("Total assets must be positive")

    return total_debt / total_assets * 100
```

For the example:

``` text
Debt ratio = 9,000,000 / 15,000,000 × 100
           = 60%
```

The definition must be agreed with the organization's finance
methodology. Different organizations may define leverage and debt ratios
differently. Do not silently substitute one formula for another.

Step 5 --- Retrieve relevant explanatory documents if needed, such as
internal financial-analysis methodology or risk definitions.

Step 6 --- Build a context containing source facts, calculated results,
definitions, and provenance.

Step 7 --- Ask the LLM to explain the result and compare periods.
Require it to distinguish measured facts from interpretation.

Step 8 --- Validate the answer against source values and return
citations or source references.

**Engineering principle:** the LLM should not be the source of truth for
exact financial values or arithmetic.

------------------------------------------------------------------------

## 6. Enterprise Challenges and Best Practices

### 6.1 Hallucination

**Problem:** the model generates information that is not supported by
the available evidence.

**Mitigations:** - Require evidence-based answers. - Provide source
references. - Define when the model must abstain. - Use APIs for
authoritative values. - Calculate numeric results in code. - Validate
critical outputs. - Evaluate with representative test questions.

### 6.2 Retrieval failure

**Problem:** relevant information exists but is not retrieved.

Possible causes: - Poor chunk boundaries - Incompatible or weak
embeddings - Incorrect metadata filters - Low Top-K - Domain terminology
mismatch - Missing query rewriting - Incorrect index configuration

Potential improvements: - Improve parsing and chunking - Test
domain-specific embedding models - Add lexical retrieval - Review
metadata filters - Tune Top-K - Add reranking - Create a retrieval
evaluation set

### 6.3 Context pollution

**Problem:** irrelevant or duplicated chunks enter the prompt.

Mitigations: - Filter by company, period, and category. - Deduplicate
overlapping results. - Rerank candidates. - Compress context
carefully. - Set a context budget. - Preserve critical qualifications
and units.

### 6.4 Stale data and lifecycle management

Store and distinguish: - Business-effective date - Reporting period -
Source update timestamp - Ingestion timestamp - Document version

Define refresh frequency, expiration, deletion, and re-indexing
procedures. A document uploaded today may describe a financial period
from several years ago.

### 6.5 Security and authorization

Enterprise RAG must enforce security at the application and data layers.

Best practices: - Authenticate every request. - Derive user and tenant
scope from trusted identity claims. - Apply authorization filters before
retrieval. - Enforce authorization again in business services. - Avoid
exposing sensitive content in logs and traces. - Encrypt data in transit
and at rest. - Apply retention and deletion policies. - Test
cross-tenant and cross-customer isolation. - Audit access to sensitive
records.

Do not rely on a prompt instruction such as "do not show other
customers' data" as an access-control mechanism.

### 6.6 Prompt injection

Retrieved documents and user input may contain malicious instructions.

Best practices: - Treat retrieved content as untrusted data. - Separate
system instructions from retrieved evidence. - Do not let retrieved text
grant permissions. - Validate tool names and arguments. - Enforce
authorization outside the LLM. - Limit tool capabilities and side
effects. - Log and review suspicious activity.

### 6.7 Reliability and resilience

Production systems should also consider: - Timeouts and bounded
retries - Circuit breakers for failing dependencies - Idempotency for
actions - Graceful degradation - Queue-based ingestion - Dead-letter
handling - Versioned prompts and indexes - Backups and recovery - Rate
limiting and concurrency controls

------------------------------------------------------------------------

## 7. RAG Evaluation and Observability

A production RAG system must be evaluated at both retrieval and
generation stages.

### 7.1 Retrieval metrics

  -----------------------------------------------------------------------
  Metric                              Meaning
  ----------------------------------- -----------------------------------
  Recall@K                            Fraction of relevant evidence
                                      retrieved within the first K
                                      results

  Precision@K                         Fraction of the first K results
                                      that are relevant

  MRR                                 Rewards placing the first relevant
                                      result near the top

  NDCG                                Measures ranking quality while
                                      considering relevance and position
  -----------------------------------------------------------------------

Example: if relevant evidence is found in the top five results for 80
out of 100 test questions:

``` text
Recall@5 = 80 / 100 = 0.80 = 80%
```

### 7.2 Generation and context metrics

  -----------------------------------------------------------------------
  Metric                              Meaning
  ----------------------------------- -----------------------------------
  Faithfulness                        Whether the answer is supported by
                                      the supplied context

  Answer relevance                    Whether the answer addresses the
                                      question

  Context precision                   How much retrieved context is
                                      relevant

  Context recall                      Whether the context contains the
                                      required evidence
  -----------------------------------------------------------------------

Frameworks such as Ragas can help evaluate RAG pipelines. Automated
scores should be supplemented with human review, especially for
financial, legal, regulatory, and other high-impact domains.

### 7.3 Operational metrics

Monitor: - End-to-end latency, including p50 and p95 - Retrieval and
reranking latency - LLM latency and token usage - Cost per request - API
and tool-call success rates - Empty or low-confidence retrieval rate -
Timeout and retry rates - Index freshness - User feedback -
Authorization failures

Use distributed tracing to connect a user request to retrieval calls,
business API calls, model calls, and validation results. Avoid recording
secrets or unnecessary sensitive data.

### 7.4 Build an evaluation dataset

Create a version-controlled dataset containing:

-   User question
-   Expected intent or route
-   Expected entities and filters
-   Relevant source documents or records
-   Expected answer facts
-   Expected abstention behavior, where applicable

Include normal, ambiguous, multi-hop, out-of-scope, stale-data, and
adversarial questions. Run the dataset after changes to chunking,
embeddings, prompts, retrieval settings, or models.

------------------------------------------------------------------------

## 8. Useful Technologies

The following are examples, not mandatory components.

  Technology      Typical role
  --------------- ---------------------------------------------------
  LangChain       Components and integrations for LLM applications
  LlamaIndex      Data ingestion, indexing, and retrieval workflows
  LangGraph       Stateful and multi-step agent workflows
  Milvus          Vector database and vector search
  Qdrant          Vector database and filtering
  Elasticsearch   Lexical, vector, and hybrid search
  Redis           Caching and fast data access
  FastAPI         Python API development
  Ragas           RAG evaluation
  Ollama          Local model execution
  vLLM            High-throughput LLM serving

Choose technologies based on requirements, operational skills, security
constraints, scale, and measured performance. Avoid adding frameworks
when a small, explicit implementation is easier to maintain.

------------------------------------------------------------------------

## 9. Practical Design Checklist

### Data and indexing

-   [ ] Identify authoritative source systems.
-   [ ] Define ingestion and refresh schedules.
-   [ ] Preserve source IDs, versions, dates, and units.
-   [ ] Normalize data without losing business meaning.
-   [ ] Select chunking strategies by content type.
-   [ ] Design metadata and access-scope fields.
-   [ ] Select and evaluate an embedding model.
-   [ ] Define re-indexing and deletion processes.

### Retrieval

-   [ ] Understand intent and extract entities.
-   [ ] Preserve company, tenant, and period constraints.
-   [ ] Apply authorization before returning evidence.
-   [ ] Tune Top-K using an evaluation dataset.
-   [ ] Evaluate vector and hybrid retrieval.
-   [ ] Add reranking if it improves measured quality.
-   [ ] Deduplicate and build a context budget.

### Generation

-   [ ] Use clear system instructions.
-   [ ] Separate facts from interpretation.
-   [ ] Require citations or source references where possible.
-   [ ] Define abstention behavior.
-   [ ] Validate structured output and numeric claims.
-   [ ] Treat retrieved content as untrusted input.

### Production

-   [ ] Add authentication and authorization.
-   [ ] Add timeouts, retries, and rate limits.
-   [ ] Monitor latency, cost, and failures.
-   [ ] Add tracing and audit logs.
-   [ ] Protect sensitive data in logs.
-   [ ] Test tenant isolation and prompt-injection defenses.
-   [ ] Maintain regression evaluation tests.

------------------------------------------------------------------------

## 10. Recommended Learning Path

A practical learning sequence for a software engineer:

1.  **RAG foundations:** LLMs, embeddings, chunking, metadata, vector
    databases, and similarity search.
2.  **Retrieval engineering:** Top-K, filtering, hybrid search, query
    rewriting, and reranking.
3.  **Enterprise data:** JSON normalization, record-aware chunking,
    source-of-truth design, data freshness, and authorization.
4.  **Orchestration and tools:** API integration, function calling,
    routing, deterministic calculations, and error handling.
5.  **Advanced architectures:** Modular RAG, Graph RAG, and Agentic RAG.
6.  **Production engineering:** evaluation, observability, latency,
    caching, security, and cost control.

Build a small end-to-end implementation first. Then improve one
component at a time and measure the impact with a fixed evaluation
dataset.

------------------------------------------------------------------------

## 11. Final Principles

1.  RAG is a complete data and application architecture, not just a
    vector database.
2.  Indexing quality strongly affects retrieval quality.
3.  Chunking should preserve meaning, structure, and provenance.
4.  Metadata is essential for filtering, traceability, and lifecycle
    management.
5.  Similarity scores are not confidence or correctness scores.
6.  Hybrid search is useful when exact terms and semantic meaning both
    matter.
7.  Reranking can improve candidate ordering but cannot recover missing
    candidates.
8.  Use authoritative APIs or databases for exact structured values.
9.  Use deterministic code for calculations and business rules.
10. Use the LLM to interpret evidence and communicate results.
11. Enforce authorization outside the LLM.
12. Evaluate retrieval, generation, security, and operations separately.

**The goal of enterprise RAG is not merely to generate fluent answers.
It is to produce useful, traceable, secure, and reliable answers from
the right business data.**
